# Feedback loops and causal diagrams in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/03-loops.ipynb)

A feedback loop is a closed chain of cause and effect: a stock changes a flow, and the flow changes the stock. This notebook shows the two kinds of loop, **reinforcing** and **balancing**, in running models, gives you a tiny function that works out a loop's type from its links, and ends with a model you write yourself in which a balancing loop quietly undoes a policy.

The ideas are taught in the app module [Feedback loops and causal diagrams](https://waelrash1.github.io/loopbricks/#/learn/loops).

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. A reinforcing loop

In **Population Growth** (`growth`), births are a fixed share of the population, so more people means more births, and more births means more people. That chain closes on itself and amplifies whatever is already happening: a **reinforcing** loop. Deaths form a second loop: more people, more deaths, fewer people. Rates are in percent a year.

In [ ]:
g = lb.molecule("growth")
print(g.equations())

fig, ax = plt.subplots(figsize=(8, 4))
for name in ("Growing", "Stable", "Declining"):
    g.run(preset=name)["Population"].plot(ax=ax, label=name)
ax.set_ylabel("Population"); ax.legend(); ax.grid(alpha=0.3)

run = g.run(preset="Growing")
for year in (0.0, 20.0, 40.0, 60.0):
    print(f"year {year:2.0f}: population {run.loc[year, 'Population']:6.1f}, births {run.loc[year, 'Births']:5.1f} a year")

In the *Growing* run (births 5%, deaths 2%), births rise from 5 a year to 30 a year because the population they come from keeps growing. Each 20 years multiplies the population by the same factor, about 1.8, which is the signature of a reinforcing loop: the change is proportional to the state. With equal rates the two loops cancel and nothing moves; with more deaths than births the same structure produces decline.

**Try it.** In the *Growing* preset the net growth is 3% a year. Using the rule of 70 (doubling time ≈ 70 / growth in percent), estimate when the population passes 1,000, then check with a longer run, `g.run(preset="Growing", T=100)`.

<details><summary>Show a worked answer</summary>

70 / 3 ≈ 23 years to double. Going from 100 to 1,000 is a factor of 10, which is a bit more than three doublings (2³ = 8, 2⁴ = 16), about 3.3 of them: 3.3 × 23 ≈ 77 years. The run passes 1,000 in year 76.9.
</details>

In [ ]:
long = g.run(preset="Growing", T=100)["Population"]
print("Population first passes 1,000 in year", long.index[long >= 1000][0])

## 2. Telling the two kinds of loop apart

Every link in a causal diagram has a sign. A **+** link means the effect moves in the same direction as the cause; a **−** link means it moves the opposite way. Go round the loop and count the minus signs: an even number (zero included) makes a reinforcing loop, an odd number a balancing one. That rule fits in four lines of Python.

In [ ]:
def loop_type(links):
    """links: a list of (cause, effect, sign) going once round the loop."""
    minus = sum(1 for cause, effect, sign in links if sign == "-")
    return "reinforcing (R)" if minus % 2 == 0 else "balancing (B)"

births_loop = [("Population", "Births", "+"), ("Births", "Population", "+")]
deaths_loop = [("Population", "Deaths", "+"), ("Deaths", "Population", "-")]   # an outflow lowers the stock
gap_loop = [("Actual", "Adjusting", "-"), ("Adjusting", "Actual", "+")]         # Adjusting = (Goal - Actual) / time
for name, loop in (("births", births_loop), ("deaths", deaths_loop), ("close gap", gap_loop)):
    print(f"{name:10} loop: {loop_type(loop)}")

cg = lb.molecule("closegap")
fig, ax = plt.subplots(figsize=(8, 4))
for goal in (60, 40, 5):
    cg.run(goal=goal)["Actual"].plot(ax=ax, label=f"goal {goal}")
ax.set_ylabel("Actual"); ax.legend(); ax.grid(alpha=0.3)

**Close Gap** (`closegap`) starts at 20 and has one balancing loop. Whether the goal is above or below the start, the stock heads for it and slows down as it gets close: a balancing loop opposes any gap, in either direction. A reinforcing loop does the reverse and makes a gap grow.

**Try it.** In **Stock management** (`stockmgmt`, print `lb.molecule("stockmgmt").equations()`), orders depend on the gap in inventory and the gap in the supply line. Write the two loops through `Orders` as lists of links and classify them.

<details><summary>Show a worked answer</summary>

Inventory loop: Inventory → Orders (−, more inventory means fewer orders), Orders → Supply line (+), Supply line → Acquisition (+), Acquisition → Inventory (+). One minus sign: balancing. Supply line loop: Supply line → Orders (−, a fuller supply line means fewer orders), Orders → Supply line (+). One minus sign: balancing too. Both loops work to close a gap, which is what a stock management rule is for.
</details>

In [ ]:
inventory_loop = [("Inventory", "Orders", "-"), ("Orders", "Supply line", "+"),
                  ("Supply line", "Acquisition", "+"), ("Acquisition", "Inventory", "+")]
supply_line_loop = [("Supply line", "Orders", "-"), ("Orders", "Supply line", "+")]
print("inventory loop:  ", loop_type(inventory_loop))
print("supply line loop:", loop_type(supply_line_loop))

## 3. A balancing loop has limits

In **Schedule Pressure** (`schedulePressure`), a team has 400 tasks and a deadline. The required rate is the work left divided by the time left; the team speeds up or slows down in proportion, but people can only work between half and twice their normal rate of 25 tasks a week. The loop runs: work remaining → required rate (+) → schedule pressure (+) → work rate (+) → work remaining (−). One minus sign, so it is balancing: it pulls the finishing date towards the deadline.

In [ ]:
sp = lb.molecule("schedulePressure")
print("deadline  work rate  finished in week")
for deadline in (25, 20, 15, 10, 8, 6):
    run = sp.run(scheduled_date=deadline)
    left = run["Work remaining"]
    print(f"{deadline:8}  {run['Work rate'].iloc[0]:9.1f}  {left.index[left < 1][0]:16.1f}")

With a loose deadline of 25 weeks the team relaxes to 16 tasks a week, and with 10 weeks it pushes to 40: the loop adapts the work rate to the deadline in both directions and the team finishes within about half a week of the target. (The last few tasks trail off over a few days, which is why every run ends slightly after its deadline.) At 8 weeks the team needs exactly twice its normal rate, the most it can give. At 6 weeks it would need 67 tasks a week, the loop hits its limit and stops working, and the project finishes at the same time as the 8-week one. A loop's strength is never unlimited, and the limit is often where the interesting behaviour is.

**Try it.** How tight can the deadline be before the team hits the limit of 2? Find it with arithmetic first, then check by running deadlines of 9, 8 and 7 weeks and printing `run["Effect of pressure"].max()`.

<details><summary>Show a worked answer</summary>

The team can do at most 2 × 25 = 50 tasks a week, so 400 tasks need at least 400 / 50 = 8 weeks. At 9 weeks the effect peaks at 1.78, at 8 weeks it reaches exactly 2, and at 7 weeks it is stuck at 2 from the start: any deadline tighter than 8 weeks will slip.
</details>

In [ ]:
for deadline in (9, 8, 7):
    print(f"deadline {deadline}: effect of pressure peaks at {sp.run(scheduled_date=deadline)['Effect of pressure'].max():.2f}")

## 4. When a balancing loop fights your policy

Balancing loops are not only in the models we build: they are in the world, and they often push back against a policy. A classic case is road building. A town's roads take 1,000 car trips an hour at a travel time of 20 minutes. In year 5 the town adds 50% more road. Travel times fall, so driving becomes more attractive, people who used to take the bus or travel at other times start driving, and trips grow until travel is as slow as people will put up with.

You can write this as a small model in the same dictionary format as the app's model builder.

In [ ]:
roads = lb.Model.from_spec({
    "name": "Roads and traffic", "timeUnit": "year", "T": 20, "dt": 0.1,
    "stocks": [{"id": "trips", "name": "Trips", "init": "1000", "x": 0, "y": 0}],
    "flows": [
        {"id": "chg", "name": "Change in trips", "to": "trips", "eq": "(Desired_trips - Trips) / Adjustment_time", "x": 0, "y": 0},
    ],
    "vars": [
        {"id": "extra", "name": "Extra capacity", "eq": "500", "x": 0, "y": 0},
        {"id": "cap", "name": "Road capacity", "eq": "1000 + STEP(Extra_capacity, 5)", "x": 0, "y": 0},
        {"id": "tt", "name": "Travel time", "eq": "20 * (Trips / Road_capacity) ^ 2", "x": 0, "y": 0},
        {"id": "ok", "name": "Acceptable travel time", "eq": "20", "x": 0, "y": 0},
        {"id": "pool", "name": "People who could drive", "eq": "3000", "x": 0, "y": 0},
        {"id": "want", "name": "Desired trips", "eq": "MIN(People_who_could_drive, Trips * Acceptable_travel_time / Travel_time)", "x": 0, "y": 0},
        {"id": "adj", "name": "Adjustment time", "eq": "2", "x": 0, "y": 0},
    ],
})
traffic = roads.run()
roads.plot(traffic, columns=["Travel time"])
print(traffic.loc[[0.0, 5.0, 6.0, 8.0, 10.0, 20.0], ["Trips", "Travel time"]].round(1))

The new road works on the day it opens: travel time drops from 20 to 8.9 minutes. Then the balancing loop (quicker trips → more trips → slower trips) takes over, and by year 10 travel time is back at 19.9 minutes with 1,500 trips an hour. The town has more traffic and the same congestion. This is **policy resistance**: the system has a goal of its own (here, the travel time people accept), and the policy only shifted the stock that the loop adjusts. To change the result you have to change that goal or the loop, not just push on the stock.

**Try it.** Suppose only 1,300 people could ever drive (`people_who_could_drive=1300`). What travel time does the town end up with after the new road?

<details><summary>Show a worked answer</summary>

Trips can now grow only to 1,300, so travel time settles at 20 × (1300 / 1500)² ≈ 15 minutes. The run gives 15.0 in year 20. When the loop runs out of people to attract, the policy keeps part of its benefit: resistance depends on how much room the loop has.
</details>

In [ ]:
capped = roads.run(people_who_could_drive=1300)
print(f"Travel time in year 20: {capped['Travel time'].iloc[-1]:.1f} minutes, with {capped['Trips'].iloc[-1]:.0f} trips an hour")

## Where next

- Back in the app: the [Feedback loops and causal diagrams](https://waelrash1.github.io/loopbricks/#/learn/loops) module covers drawing loops so other people can read them, with more examples of policy resistance.
- Next notebook: [Patterns of behaviour and the structures behind them](https://waelrash1.github.io/loopbricks/#/learn/patterns), which connects each pattern of behaviour to the loops that produce it. [First-order feedback](https://waelrash1.github.io/loopbricks/#/learn/first-order) solves the single loops of sections 1 and 2 exactly.
- The surprises in [Why complex systems surprise us](https://waelrash1.github.io/loopbricks/#/learn/surprise) are balancing loops and delays at work.